# Automated Seat Belt Detection — YOLOv8 + CLAHE

End-to-end Colab notebook: install → download dataset → CLAHE preprocess → train YOLOv8 → evaluate → inference → Twilio alert.

**Runtime:** set `Runtime → Change runtime type → GPU` (T4 is fine).

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow twilio opencv-python python-dotenv

## 2. (Optional) Mount Google Drive
Use this to persist weights/datasets across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Download the dataset (Roboflow)

Get a free API key at https://app.roboflow.com/settings/api .
Store it as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left sidebar), or paste it below.

Default dataset: `bassant/seatbeltfinal`. Swap in any seat-belt dataset: on its Roboflow Universe page click **Download Dataset → YOLOv8 → show download code**.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    os.environ.setdefault('ROBOFLOW_API_KEY', 'PASTE_YOUR_KEY_HERE')

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
project = rf.workspace('bassant').project('seatbeltfinal')
dataset = project.version(1).download('yolov8')
DATA_DIR = dataset.location
print('Dataset at:', DATA_DIR)
print('data.yaml:')
print(open(os.path.join(DATA_DIR, 'data.yaml')).read())

## 4. CLAHE preprocessing

Enhance local contrast (helps in low light). Creates a CLAHE copy of the dataset; labels are copied unchanged.

In [ ]:
import cv2, shutil
from pathlib import Path

def apply_clahe(img, clip_limit=2.0, tile=(8,8)):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

src_root = Path(DATA_DIR)
dst_root = Path(DATA_DIR + '_clahe')
exts = {'.jpg','.jpeg','.png','.bmp','.webp'}
n = 0
for split in ('train','valid','val','test'):
    sdir = src_root/split
    if not sdir.is_dir():
        continue
    (dst_root/split/'images').mkdir(parents=True, exist_ok=True)
    (dst_root/split/'labels').mkdir(parents=True, exist_ok=True)
    for img in sorted((sdir/'images').glob('*')):
        if img.suffix.lower() not in exts:
            continue
        im = cv2.imread(str(img))
        if im is None:
            continue
        cv2.imwrite(str(dst_root/split/'images'/img.name), apply_clahe(im)); n += 1
    for lbl in (sdir/'labels').glob('*.txt'):
        shutil.copy2(lbl, dst_root/split/'labels'/lbl.name)
shutil.copy2(src_root/'data.yaml', dst_root/'data.yaml')
print(f'CLAHE done: {n} images -> {dst_root}')

## 5. Train YOLOv8

`yolov8s.pt` for best accuracy, `yolov8n.pt` for fastest inference.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
results = model.train(
    data=str(dst_root/'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    name='seatbelt_yolov8',
    patience=20,
)
print('Best weights:', results.save_dir)

## 6. Evaluate

In [ ]:
metrics = model.val()
print('mAP@0.5     :', metrics.box.map50)
print('mAP@0.5:0.95:', metrics.box.map)
print('Precision   :', metrics.box.mp)
print('Recall      :', metrics.box.mr)

## 7. Inference on a sample image

In [ ]:
import glob
from ultralytics import YOLO
best = str(results.save_dir) + '/weights/best.pt'
model = YOLO(best)

sample = sorted(glob.glob(str(dst_root/'valid'/'images'/'*')))[0]
img = apply_clahe(cv2.imread(sample))          # same CLAHE as training
res = model.predict(img, conf=0.35, verbose=False)[0]

from google.colab.patches import cv2_imshow
cv2_imshow(res.plot())
labels = [model.names[int(b.cls[0])] for b in res.boxes]
print('Detected:', labels)

## 8. Twilio SMS alert

Store `TWILIO_ACCOUNT_SID`, `TWILIO_AUTH_TOKEN`, `TWILIO_FROM_NUMBER` as Colab secrets. The alert fires only when a 'no seat belt' class is detected.

In [ ]:
from twilio.rest import Client
from google.colab import userdata

VIOLATION = {'no_seatbelt','no-seatbelt','no','without_seatbelt','0'}

def send_alert(to_number, vehicle):
    sid  = userdata.get('TWILIO_ACCOUNT_SID')
    tok  = userdata.get('TWILIO_AUTH_TOKEN')
    frm  = userdata.get('TWILIO_FROM_NUMBER')
    client = Client(sid, tok)
    msg = client.messages.create(
        body=f'\u26a0\ufe0f Alert: Seat belt not detected for vehicle {vehicle}. Please ensure driver safety.',
        from_=frm, to=to_number)
    print('SMS sent:', msg.sid)

detected = {l.lower() for l in labels}
if detected & VIOLATION:
    send_alert('+911234567890', 'AP 39 XX 1234')
else:
    print('No violation detected — no alert sent.')